# Customer Churn Prediction
**Auto-adaptive — works with any churn dataset**

1. Install & import
2. Load data
3. EDA
4. Train — auto-selects best algorithm
5. Feature importances
6. Predict on new customers

> **To use your own data:** replace the sample DataFrame below, update `TARGET_COL` and `DROP_COLS`.

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn --quiet
print('Done.')

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from models.churn.churn_model import ChurnModel
print('Imports ready.')

In [ ]:
# CONFIG
TARGET_COL = 'churned'
DROP_COLS  = ['customer_id']

# DATA — swap for: df = pd.read_csv('../data/sample_datasets/your_file.csv')
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'customer_id':      range(n),
    'tenure_months':    np.random.randint(1, 72, n),
    'monthly_charges':  np.round(np.random.uniform(20, 120, n), 2),
    'total_charges':    np.round(np.random.uniform(100, 8000, n), 2),
    'num_products':     np.random.randint(1, 6, n),
    'support_calls':    np.random.randint(0, 10, n),
    'contract_type':    np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], n),
    'payment_method':   np.random.choice(['Credit Card', 'Bank Transfer', 'Electronic Check'], n),
    'internet_service': np.random.choice(['DSL', 'Fiber', 'No'], n),
    'churned':          np.random.choice([0, 1], n, p=[0.73, 0.27])
})
print(f'Shape: {df.shape} | Churn rate: {df[TARGET_COL].mean():.1%}')
df.head()

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=['int64','float64']).columns if c not in [TARGET_COL]+DROP_COLS]
if numeric_cols:
    fig, axes = plt.subplots(1, len(numeric_cols), figsize=(5*len(numeric_cols), 4))
    if len(numeric_cols) == 1: axes = [axes]
    for ax, col in zip(axes, numeric_cols):
        for label, grp in df.groupby(TARGET_COL):
            grp[col].plot(kind='kde', ax=ax, label=f'Churn={label}')
        ax.set_title(col); ax.legend()
    plt.suptitle('Numeric Features: Churned vs Retained', fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()

In [ ]:
cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in DROP_COLS]
if cat_cols:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(5*len(cat_cols), 4))
    if len(cat_cols) == 1: axes = [axes]
    overall = df[TARGET_COL].mean()
    for ax, col in zip(axes, cat_cols):
        df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False).plot(kind='bar', ax=ax, color='#e74c3c', alpha=0.8)
        ax.axhline(overall, linestyle='--', color='black', lw=1, label=f'Avg {overall:.1%}')
        ax.set_title(f'Churn Rate by {col}'); ax.legend()
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    plt.suptitle('Churn Rate by Category', fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()

In [ ]:
corr = df[numeric_cols + [TARGET_COL]].corr()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Correlation Matrix'); plt.tight_layout(); plt.show()

In [ ]:
model = ChurnModel(target_col=TARGET_COL, drop_cols=DROP_COLS)
model.fit(df)

In [ ]:
model.feature_importance(top_n=15)

In [ ]:
new_customers = pd.DataFrame({
    'customer_id':      [9001, 9002, 9003, 9004],
    'tenure_months':    [2, 36, 60, 1],
    'monthly_charges':  [95.0, 45.0, 30.0, 110.0],
    'total_charges':    [190.0, 1620.0, 1800.0, 110.0],
    'num_products':     [1, 3, 5, 1],
    'support_calls':    [8, 1, 0, 9],
    'contract_type':    ['Month-to-Month','One Year','Two Year','Month-to-Month'],
    'payment_method':   ['Electronic Check','Credit Card','Bank Transfer','Electronic Check'],
    'internet_service': ['Fiber','DSL','DSL','Fiber']
})
predictions = model.predict(new_customers)
pd.concat([new_customers[['customer_id','tenure_months','contract_type']], predictions], axis=1)